# See how adjudicator's behaviour

This notebook computes PRP and RA summaries in memory from merged benchmark files.

No summary CSVs are created and figures are shown inline only.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
# repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()


In [3]:
# Set run_id explicitly, or keep None to use latest run folder.
ignore_models = ['smollm2-1.7b', 'gemma3-4b', 'phi3-3.8b', 'deepseek-v3.2-cloud','qwen3.5-397b-cloud']
run_id =None
# run_id = "20260422_212121"
run_id = "20260726_134607_1"
# run_id = "20260517_175302"
# run_id = "20260413_151017"

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
bench_root = repo_root / "data" / "results" / "affinity_benchmark"

if run_id is None:
    run_dirs = sorted([p for p in bench_root.iterdir() if p.is_dir()])
    if not run_dirs:
        raise FileNotFoundError(f"No benchmark runs found under {bench_root}")
    run_dir = run_dirs[-1]
else:
    run_dir = bench_root / run_id

prp_path = run_dir / "merged_prp_affinities.csv"
ra_path = run_dir / "merged_ra_affinities.csv"
prp_adjudication_path = run_dir / "adjudicated_prp_affinities.csv"
ra_adjudication_path = run_dir / "adjudicated_ra_affinities.csv"


prp_df = pd.read_csv(prp_path)
prp_df = prp_df[~prp_df['Model_Slug'].isin(ignore_models)]

print("Run directory:", run_dir)
print("PRP rows:", len(prp_df), "| Models:", prp_df['Model_Slug'].nunique())
if ra_path.exists():
    ra_df = pd.read_csv(ra_path)
    ra_df = ra_df[~ra_df['Model_Slug'].isin(ignore_models)]
    print("RA rows:", len(ra_df), "| Models:", ra_df['Model_Slug'].nunique())

prp_adjudication_df = pd.read_csv(prp_adjudication_path)
ra_adjudication_df = pd.read_csv(ra_adjudication_path)




Run directory: /mnt/c/Users/pigal/OneDrive - Imperial College London/PhD/Repositories/article-classifier/data/results/affinity_benchmark/20260726_134607_1
PRP rows: 3675 | Models: 5
RA rows: 25200 | Models: 5


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/c/Users/pigal/OneDrive - Imperial College London/PhD/Repositories/article-classifier/data/results/affinity_benchmark/20260726_134607_1/adjudicated_prp_affinities.csv'

In [ ]:

# Merge prp_adjudication_df into prp_df
# Step 1: Rename columns
prp_adj_merged = prp_adjudication_df.copy()
prp_adj_merged = prp_adj_merged.rename(columns={
    'Final_Affinity': 'LLM_Affinity',
    'Final_Reason': 'LLM_Affinity_Reason'
})

# Step 2: Extract the base model name and remove "_adjudicator" suffix
non_null_slugs = prp_adj_merged['Model_Slug'].dropna().unique()
if len(non_null_slugs) > 0:
    base_model = str(non_null_slugs[0])+'_adjudicator'
    print(f"Base model from adjudication: {base_model}")

prp_adj_merged["Model_Slug"] = base_model

prp_df = pd.concat([prp_df, prp_adj_merged[prp_df.columns]], ignore_index=True)
# Merge ra_adjudication_df into ra_df
ra_adj_merged = ra_adjudication_df.copy().rename(
    columns={
        "Final_Affinity": "LLM_Affinity",
        "Final_Reason": "LLM_Affinity_Reason",
        "Cosine_x100_Mean": "Cosine_x100",
    }
)

non_null_ra_slugs = ra_adj_merged["Model_Slug"].dropna().unique()
if len(non_null_ra_slugs) > 0:
    base_model_ra = f"{non_null_ra_slugs[0]}_adjudicator"
else:
    base_model_ra = "adjudicator"

print(f"Base model from RA adjudication: {base_model_ra}")
ra_adj_merged["Model_Slug"] = base_model_ra

for col in ra_df.columns:
    if col not in ra_adj_merged.columns:
        ra_adj_merged[col] = pd.NA

ra_df = pd.concat([ra_df, ra_adj_merged[ra_df.columns]], ignore_index=True)
print("RA rows after merge:", len(ra_df), "| Models:", ra_df["Model_Slug"].nunique())




In [ ]:
mapping_path = repo_root / 'data' / 'processed' / 'ra_grouping_ra2025.csv'
ra_grouping_df = pd.read_csv(mapping_path)

required_cols = {'RA2025', 'Grouping'}
missing_cols = required_cols - set(ra_grouping_df.columns)
if missing_cols:
    raise ValueError(f'Missing required columns in mapping file: {missing_cols}')

ra_to_prp_map = (
    ra_grouping_df[['RA2025', 'Grouping']]
    .dropna(subset=['RA2025', 'Grouping'])
    .drop_duplicates()
    .rename(columns={'RA2025': 'RA2025_ID', 'Grouping': 'PRP_Name'})
)
ra_to_prp_map['RA2025_ID'] = pd.to_numeric(ra_to_prp_map['RA2025_ID'], errors='coerce').astype('Int64')

print('Mapping file:', mapping_path)
print('Mapping rows:', len(ra_to_prp_map))
display(ra_to_prp_map.head(10))

ra_df = ra_df.merge(
    ra_to_prp_map,
    on='RA2025_ID',
    how='left'
 )

# PRP
Affinity level of an **abstract** to each **primary research programme** (PRP)

In [ ]:
fig = px.box(
    prp_df,
    color='Model_Slug',
    y='LLM_Affinity',
    x = 'PRP_Name',
    # points = 'all',
    title='Affinity Distribution by PRP across models',
 )
# fig.update_layout(xaxis_title='PRP', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# top_disaggrement_ids = (prp_adjudication_df[prp_adjudication_df['agreement_mean']<1]).nlargest(5, 'Affinity_Range')['Abstract_Index']
# low_disagreements_ids = (prp_adjudication_df[prp_adjudication_df['agreement_mean']==1]).nsmallest(5, 'Affinity_Range')['Abstract_Index']
# fig = px.box(
#     prp_df[prp_df['Abstract_Index'].isin(top_disaggrement_ids)], #ra_df[ra_df['Abstract_Index'].isin([7,8,9])],
#     # x='Abstract_Index',
#     y='LLM_Affinity',
#     # color='Model_Slug',
#     points='all',
#     hover_data='PRP_Name',
#     # facet_row='PRP_Name',
#     facet_col='Abstract_Index',
#     color = 'PRP_Name',
#     # x = 'Model_Slug',
#     title=f'PRP Affinity Distribution by Abstract'
#  )
# # fig.update_xaxes(range=[0.5,10.5])
# fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# # fig.update_xaxes(tickangle=45)
# fig.show()

# fig = px.box(
#     prp_df[prp_df['Abstract_Index'].isin(low_disagreements_ids)], #ra_df[ra_df['Abstract_Index'].isin([7,8,9])],
#     # x='Abstract_Index',
#     y='LLM_Affinity',
#     # color='Model_Slug',
#     points='all',
#     hover_data='PRP_Name',
#     # facet_row='PRP_Name',
#     facet_col='Abstract_Index',
#     color = 'PRP_Name',
#     # x = 'Model_Slug',
#     title=f'PRP Affinity Distribution by Abstract'
#  )
# # fig.update_xaxes(range=[0.5,10.5])
# fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# # fig.update_xaxes(tickangle=45)
# fig.show()



In [ ]:
top_disaggrement_ids = (prp_adjudication_df[prp_adjudication_df['agreement_mean']<1]).nlargest(5, 'Affinity_Range')['Abstract_Index']
low_disagreements_ids = (prp_adjudication_df[prp_adjudication_df['agreement_mean']==1]).nsmallest(5, 'Affinity_Range')['Abstract_Index']
fig = px.scatter(
    prp_df[prp_df['Abstract_Index'].isin(top_disaggrement_ids)], #ra_df[ra_df['Abstract_Index'].isin([7,8,9])],
    # x='Abstract_Index',
    y='LLM_Affinity',
    # color='Model_Slug',
    # points='all',
    # hover_data='PRP_Name',
    # facet_row='PRP_Name',

    facet_col='Abstract_Index',
    color = 'Model_Slug',
    x = 'PRP_Name',
    # title=f'PRP Affinity Distribution by Abstract',
    # markers = True
 )
# fig.update_xaxes(range=[0.5,10.5])
fig.update_layout(yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
pd.set_option("display.max_colwidth", None)

_preview = (
    prp_adjudication_df.loc[prp_adjudication_df["Abstract_Index"].isin(top_disaggrement_ids)]
    .sort_values(["Abstract_Index", "Affinity_Range", "agreement_mean"], ascending=[True, False, True])
    .drop_duplicates(subset=["Abstract_Index"], keep="first")
    [["Abstract_Index", "PRP_Name", "Final_Reason"]]
)

print("\nFull Final_Reason text by abstract:\n")
for row in _preview.itertuples(index=False):
    print(f"Abstract {row.Abstract_Index} | PRP: {row.PRP_Name}")
    print(f"Final Reason: {row.Final_Reason if pd.notna(row.Final_Reason) else 'N/A'}")
    print("-" * 120)
top_disagreement_prp = (
    prp_adjudication_df[
        prp_adjudication_df["Abstract_Index"].isin(top_disaggrement_ids)
    ]
    .sort_values(
        ["Abstract_Index", "Affinity_Range", "agreement_mean"],
        ascending=[True, False, True],
    )
    .groupby("Abstract_Index", as_index=False)
    .head(1)[
        [
            "Abstract_Index",
            "Document Title",
            "PRP_Name",
            "Final_Affinity",
            "Final_Reason",
            "Affinity_Range",
            "agreement_mean",
        ]
    ]
    .rename(columns={"Final_Reason": "Final Reason"})
    .reset_index(drop=True)
)

display(top_disagreement_prp)

In [ ]:
fig = px.line(
    prp_df[prp_df['Abstract_Index'].isin(top_disaggrement_ids)], #ra_df[ra_df['Abstract_Index'].isin([7,8,9])],
    # x='Abstract_Index',
    y='LLM_Affinity',
    # color='Model_Slug',
    # points='all',
    # hover_data='PRP_Name',
    # facet_row='PRP_Name',

    facet_col='Abstract_Index',
    color = 'PRP_Name',
    x = 'Model_Slug',
    title=f'PRP Affinity Distribution by Abstract',
    markers = True
 )
# fig.update_xaxes(range=[0.5,10.5])
fig.update_layout(yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# Patch px.box so every new box figure gets affinity threshold guides
id_ = 8
filter_ = prp_df['Model_Slug'] != 'gpt-5.4_adjudicator'
base_df = prp_df[(prp_df['Abstract_Index'] == id_) & filter_]
adj_df = prp_df[(prp_df['Abstract_Index'] == id_) & (~filter_)]

fig = px.box(
    base_df,
    y='LLM_Affinity',
    points='all',
    x='PRP_Name',
    color='PRP_Name',
    title=f'PRP Affinity Distribution for Abstract {id_}',
)

# Overlay adjudicator-only points as scatter
fig.add_scatter(
    x=adj_df['PRP_Name'],
    y=adj_df['LLM_Affinity'],
    mode='markers',
    name='gpt-5.4_adjudicator',
    marker=dict(size=5, symbol='x', color='purple'),
    hovertext=adj_df['PRP_Name'],
    hovertemplate='PRP_Name=%{x}<br>LLM_Affinity=%{y}<br>PRP=%{hovertext}<extra></extra>',
)
fig.update_layout(xaxis_title=None, yaxis_title='LLM Affinity', legend = dict(title=None, orientation='h', yanchor='bottom', y=-0.45, xanchor='right', x=1))
# fig.update_xaxes(tickangle=45)
fig.update_layout(width=500, height=500)
fig.show()

## RA

In [ ]:
fig = px.box(
    ra_df,
    x='RA2025_ID',
    y='LLM_Affinity',
    color='Model_Slug',
    # points='all',
    # facet_row='PRP_Name',
    # color = 'PRP_Name',
    title=f'RA Affinity Distribution by Abstract'
 )
fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()


In [ ]:
top_ra_disaggrement_ids = (ra_adjudication_df[ra_adjudication_df['agreement_mean']<1]).nlargest(5, 'Affinity_Range')['Abstract_Index']
fig = px.scatter(#prp_df[prp_df['Abstract_Index'].isin([1,8,20,19])]
    ra_df[ra_df['Abstract_Index'].isin(top_ra_disaggrement_ids)],
    # ra_df,
    x='RA2025_ID',
    y='LLM_Affinity',
    color='Model_Slug',
    facet_col = 'Abstract_Index',
    # points='all',
    hover_data = 'Model_Slug',
    # facet_row='PRP_Name',
    # color = 'PRP_Name',
    # title=f'RA Affinity Distribution by Abstract'
 )
fig.update_layout(xaxis_title='RA question', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
low_ra_disaggrement_ids = (ra_adjudication_df[ra_adjudication_df['agreement_mean']==1]).nlargest(10, 'Affinity_Range')['Abstract_Index']
fig = px.scatter(#prp_df[prp_df['Abstract_Index'].isin([1,8,20,19])],
    ra_df[ra_df['Abstract_Index'].isin(low_ra_disaggrement_ids)],
    # ra_df,
    x='RA2025_ID',
    y='LLM_Affinity',
    color='Model_Slug',
    facet_col = 'Abstract_Index',
    # points='all',
    hover_data = 'Model_Slug',
    # facet_row='PRP_Name',
    # color = 'PRP_Name',
    # title=f'RA Affinity Distribution by Abstract'
 )
fig.update_layout(xaxis_title='RA question', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
if not hasattr(px, "_original_box"):
    px._original_box = px.box

def _box_with_affinity_guides(*args, **kwargs):
    fig = px._original_box(*args, **kwargs)

    # horizontal separators
    fig.add_hline(y=40, line_dash="dash", line_color="gray")
    fig.add_hline(y=70, line_dash="dash", line_color="gray")

    # optional background bands
    fig.add_hrect(y0=0, y1=40, fillcolor="red", opacity=0.06, line_width=0)
    fig.add_hrect(y0=40, y1=70, fillcolor="orange", opacity=0.06, line_width=0)
    fig.add_hrect(y0=70, y1=100, fillcolor="green", opacity=0.06, line_width=0)

    # range labels
    # fig.add_annotation(x=1.11, y=20, xref="paper", yref="y", text="Low ", showarrow=False)
    # fig.add_annotation(x=1.18, y=55, xref="paper", yref="y", text="Medium ", showarrow=False)
    # fig.add_annotation(x=1.11, y=85, xref="paper", yref="y", text="High", showarrow=False)

    return fig

px.box = _box_with_affinity_guides

In [ ]:
ra_df['Model_Slug'].unique() 

In [ ]:
ra_df['Model_Slug'] != 'gpt-5.4_adjudicator'

In [ ]:
id_ = 98
filter_ = ra_df['Model_Slug'] != 'gpt-5.4_adjudicator'
base_df = ra_df[(ra_df['Abstract_Index'] == id_) & filter_]
adj_df = ra_df[(ra_df['Abstract_Index'] == id_) & (~filter_)]

fig = px.box(
    base_df,
    y='LLM_Affinity',
    points='all',
    x='RA2025_ID',
    color='PRP_Name',
    title=f'Questions Affinity Distribution for Abstract {id_}',
    category_orders={"PRP_Name": prp_df['PRP_Name'].unique().tolist()}
)

# Overlay adjudicator-only points as scatter
fig.add_scatter(
    x=adj_df['RA2025_ID'],
    y=adj_df['LLM_Affinity'],
    mode='markers',
    name='gpt-5.4_adjudicator',
    marker=dict(size=5, symbol='x', color='purple'),
    hovertext=adj_df['PRP_Name'],
    hovertemplate='RA2025_ID=%{x}<br>LLM_Affinity=%{y}<br>PRP=%{hovertext}<extra></extra>',
)
fig.update_layout(xaxis_title='RA Question', yaxis_title='LLM Affinity', legend = dict(title=None, orientation='h', yanchor='bottom', y=-0.4, xanchor='right', x=1))
# fig.update_xaxes(tickangle=45)
fig.update_layout(width=500, height=500)
fig.show()